# A.1: Compression as a function of vocabulary size

Train byte-level BPE tokenizers of six sizes on 90 percent of tiny Shakespeare and measure them on the rest.


In [ ]:
import os, urllib.request
from collections import Counter
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

if not os.path.exists("shakespeare.txt"):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, "shakespeare.txt")
text = open("shakespeare.txt", encoding="utf-8").read()
split = int(0.9 * len(text))
train, held_out = text[:split], text[split:]
with open("train.txt", "w", encoding="utf-8") as f:
    f.write(train)
n_bytes = len(held_out.encode("utf-8"))

for V in [512, 1000, 2000, 4000, 8000, 16000]:
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=V, show_progress=False,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train(["train.txt"], trainer)
    counts = Counter(tok.encode(train).ids)
    rare = sum(1 for i in range(tok.get_vocab_size()) if counts[i] < 10)
    n_tokens = len(tok.encode(held_out).ids)
    print(V, n_tokens, round(n_bytes / n_tokens, 2), rare)


**Expected output**

Output:

```text
512 59401 1.88 193
1000 49650 2.25 223
2000 43755 2.55 373
4000 38542 2.89 850
8000 35070 3.18 5291
16000 33720 3.31 13917
```
